# Sinais normalizados do dump Argus

## Identidade natural e escopo

Uma métrica isolada não identifica uma série: `error_rate`, por exemplo, pode existir em várias origens e entidades. Por isso, toda leitura usa a identidade completa **source, entity, metric** e uma janela explícita `[start, end)` em UTC.

Essa regra evita misturar séries parecidas, comparar origens diferentes como se fossem a mesma coisa e interpretar tempos fora do recorte real observado no dump.

In [ ]:
import plotly.graph_objects as go
from IPython.display import display
from ipywidgets import Dropdown, Output

from dump_explorer.charts import (
    METRIC_DIRECTION,
    correlation_heatmap,
    metric_timeline,
    source_metric_heatmap,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.signals import (
    list_series,
    load_correlation_frame,
    load_metric_series,
    load_source_distribution,
)

import importlib
import dump_explorer.signals as signals
import dump_explorer.quality as quality
import dump_explorer.charts as charts
importlib.reload(signals)
importlib.reload(quality)
importlib.reload(charts)

# A janela precisa vir de uma cobertura real já observada no dump.
# Usamos load_source_coverage porque ela resume, por source, os primeiros e
# últimos timestamps disponíveis em signal_bucket sem varrer o histórico inteiro.
# start_ms é inclusivo e end_ms é exclusivo: [start_ms, end_ms).
from dump_explorer.quality import load_source_coverage
import pandas as pd


settings = load_settings()
engine = create_db_engine(settings)

s = load_source_coverage(engine)
if s.empty or s['first_ts'].dropna().empty:
    raise RuntimeError("source_coverage vazio — verifique conexão/importe o dump")
start_ms = int(s['first_ts'].dropna().min().value // 10**6)
end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
print("start_ms=", start_ms, "end_ms=", end_ms)
print("start (UTC) =", pd.to_datetime(start_ms, unit='ms', utc=True))
print("end   (UTC) =", pd.to_datetime(end_ms-1, unit='ms', utc=True))
print("Janela usada no notebook: intervalo fechado-aberto [start_ms, end_ms)")
# use the safe derived window above when exploring series
# do not hardcode arbitrary dates unless you want to force a narrower recorte

## 1. Distribuição por source, entity e metric

A primeira visualização mostra quais séries existem dentro da janela escolhida e como elas se distribuem por origem. Ela ajuda a responder três perguntas: quais `source` aparecem, quais `entity` existem dentro de cada origem e quais `metric` estão disponíveis para análise.

O quadro impresso por `display(source_distribution)` resume a cobertura por origem. O heatmap em `source_metric_heatmap(series_catalog)` compacta esse catálogo em uma matriz visual para destacar concentração de séries, diversidade por origem e eventuais lacunas.

Se nada aparecer, normalmente significa que a janela não cobre dados úteis para o recorte selecionado ou que a origem não possui séries naquele intervalo.

In [ ]:
source_distribution = load_source_distribution(engine, start_ms, end_ms)
display(source_distribution)

selected_source = source_distribution.iloc[0]["source"] if not source_distribution.empty else None
series_catalog = (
    list_series(engine, selected_source, start_ms, end_ms) if selected_source is not None else None
)
if series_catalog is not None:
    display(series_catalog)
    source_metric_heatmap(series_catalog).show()

## 2. Seletor interativo de série

O seletor combina `source`, `entity` e `metric` porque a mesma métrica pode existir em vários contextos. Selecionar só o nome da métrica seria ambíguo e poderia carregar a série errada.

O gráfico de linha abaixo mostra a evolução temporal dos valores daquela identidade completa. Os buracos no tempo significam ausência de observação na janela, não zero. Já a tabela de `head()` mostra os primeiros registros carregados para confirmar que a série escolhida é a correta.

In [ ]:
series_options = [] if series_catalog is None else [
    (f"{row.source} | {row.entity} | {row.metric}", (row.source.strip(), row.entity.strip(), row.metric.strip()))
    for row in series_catalog.itertuples(index=False)
    if row.source.strip() and row.entity.strip() and row.metric.strip()  # Filtra valores vazios
]
series_selector = Dropdown(options=series_options, description="Série:")
series_output = Output()
display(series_selector, series_output)

def render_selected_series(change=None):
    if series_selector.value is None:
        return
    source, entity, metric = series_selector.value
    selected_series = load_metric_series(engine, source, entity, metric, start_ms, end_ms)
    with series_output:
        series_output.clear_output(wait=True)
        metric_timeline(selected_series, metric=metric).show()
        display(selected_series.head())
    return selected_series

series_selector.observe(render_selected_series, names="value")

# Carrega a série inicial de forma defensiva
selected_series = None
try:
    if series_selector.value is not None:
        selected_series = render_selected_series()
except ValueError as e:
    print(f"⚠️  Não foi possível carregar a série inicial: {e}")
    selected_series = None

## 3. Volume em 15 minutos e distribuição de valores

O primeiro gráfico conta quantas observações caíram em cada bloco de 15 minutos. Ele mede volume de coleta, não média nem soma de valores.

O histograma mostra a distribuição dos valores observados naquela série. Ele ajuda a perceber faixas típicas, outliers e valores muito concentrados. Se a série for percentual, uma fila de valores perto de 0 ou 100 pode ser esperada; se for latência, a escala pode ser completamente diferente.

In [ ]:
if selected_series is not None:
    volume_15min = (
        selected_series.assign(bucket_15min=selected_series["timestamp"].dt.floor("15min"))
        .groupby("bucket_15min", as_index=False)
        .size()
    )
    go.Figure(go.Bar(x=volume_15min["bucket_15min"], y=volume_15min["size"])).update_layout(
        title="Volume de observações por 15 minutos", xaxis_title="UTC", yaxis_title="Observações"
    ).show()
    go.Figure(go.Histogram(x=selected_series["value"])).update_layout(
        title="Distribuição dos valores observados", xaxis_title="Valor", yaxis_title="Frequência"
    ).show()

## 4. Direção de degradação

`METRIC_DIRECTION` é metadado semântico: ele diz se uma métrica piora quando sobe, piora quando desce ou depende do contexto. Isso não altera os valores armazenados; apenas orienta a interpretação do gráfico.

Exemplos comuns:
- `lower_is_worse`: valores maiores são melhores e quedas indicam piora;
- `higher_is_worse`: valores maiores indicam piora;
- `context_dependent`: não há regra universal só pelo nome da métrica.

In [ ]:
display(METRIC_DIRECTION)

## 5. Correlação de séries selecionadas explicitamente

Esta etapa compara séries escolhidas de forma explícita para evitar misturar identidades diferentes. A correlação usa a mesma janela UTC e ignora faltas de observação; ela não preenche valores ausentes com zero.

O gráfico `correlation_heatmap(correlation_frame)` transforma a matriz de correlação em cor para facilitar a leitura de relações fortes, fracas ou ausentes entre as séries. O limite de até 20 séries existe para evitar gráficos e cálculos excessivamente caros. Se a lista estiver vazia, o notebook apenas avisa, porque não existe correlação significativa sem pelo menos uma identidade selecionada.

In [ ]:
correlation_keys = [
    # ("newrelic", "service-name", "error_rate_pct"),
    # ("prometheus", "service-name", "latency_p95"),
]

if not correlation_keys:
    print("⚠️  correlation_keys está vazio. Descomente ou adicione identidades completas (source, entity, metric).")
else:
    correlation_frame = load_correlation_frame(engine, correlation_keys, start_ms, end_ms)
    display(correlation_frame)
    if not correlation_frame.empty:
        correlation_heatmap(correlation_frame).show()
    else:
        print("⚠️  Nenhum dado encontrado para as séries especificadas.")